# Meetup 4 Workshop: Clean Data, Zero Leakage & Your First Honest Baseline

Welcome to **Meetup 4**! Before today's session, I checked what is currently **pushed to GitHub** in all 5 repositories.

> **Already did more work on your laptop?** Awesome! Just run `git add`, `git commit`, and `git push` today so your teammates and mentor can see your latest progress.

Today's goal is simple: make sure every team avoids silent data bugs, avoids Data Leakage, and has a **clean, logged Baseline score** (or moves into Sprint 4 validation and feature engineering if your Baseline is already done!).

### Session Plan (90 minutes)
*   **18:00 - 18:10:** Quick check-in - what is on your laptop vs. GitHub right now?
*   **18:10 - 18:25:** **This short walkthrough** - 3 beginner-friendly ML rules that apply to all 5 projects.
*   **18:25 - 19:20:** Team work in your own notebook + mentor rounds (~10 min per table).
*   **19:20 - 19:30:** **Show & Tell** - each team shares 1 concrete result or lesson from today.

In [ ]:
# Run this cell once if any library is missing in VS Code / Jupyter
%pip install -q numpy pandas matplotlib scikit-learn

## Rule 1: Always Check Your Data Before Calling `.fit()`

Machine learning models do not complain when data is subtly wrong - they just learn nonsense. Before training any model, always check 4 things:
1.  **Broken files or columns:** Run `df.head()` and `df.columns`. A wrong CSV separator or shifted header can quietly put target numbers into the wrong column.
2.  **Duplicates:** If identical rows or texts appear in both `train` and `test`, the model memorizes them instead of learning. Drop duplicates *before* splitting.
3.  **Class balance (`value_counts()`):** Check how many examples you have for each label. If one class has 0 rows or only 15 rows out of 15,000, the model cannot learn it properly.
4.  **Reproducibility:** If big data files are in `.gitignore`, make sure every teammate can run the same script or load the same cached sample.

## Rule 2: What Is Data Leakage (and Why Is It Dangerous)?

**Data Leakage** means information from the **test set** or the **future** accidentally sneaks into training. It is like seeing the exam answer key while studying: you get `99%` in your notebook, and fail completely on real users!

### The 4 Leakage Traps Across the 5 Projects
*   **Time Leakage (Teams 1, 2, 5):** Never shuffle dates randomly (`train_test_split(..., shuffle=True)`). Always split **chronologically**: past -> `train`, future -> `test`.
*   **Same-Day Leakage (Teams 1, 5):** Never predict *today's* target using *today's* closing price. Shift the target up by 1 row (`shift(-1)`) so today's features predict **tomorrow**.
*   **Vocabulary / Scaling Leakage (All ML Teams):** Never fit a `StandardScaler` or `TfidfVectorizer` on the whole table before splitting. Put them inside a scikit-learn **`Pipeline`** so they learn *only* from `train`.
*   **Evaluation Trap Without `y` (Team 3):** Checking clusters only against automatic API tags (like OpenAlex labels) does not prove human usefulness. Verify a sample manually as a team and test on more than 1 topic!

In [ ]:
# TINY VISUAL DEMO: Why Random Split Lies on Time Series (Teams 1, 2, 5)
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')
np.random.seed(42)

# Simple rising time series (200 days)
days = np.arange(200)
values = 100 + 0.5 * days + 8 * np.sin(days / 10) + np.random.normal(0, 2, 200)
X, y = pd.DataFrame({'day': days}), values

# WRONG: Random split (leaks neighboring future days!)
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
pred_bad = RandomForestRegressor(random_state=42).fit(X_tr_b, y_tr_b).predict(X_te_b)

# RIGHT: Chronological split (first 80% past -> last 20% future)
cut = 160
pred_ok = RandomForestRegressor(random_state=42).fit(X[:cut], y[:cut]).predict(X[cut:])

print(f"WRONG (Random Split) MAE:        {mean_absolute_error(y_te_b, pred_bad):.2f}  <-- Fake! Peeks at future")
print(f"RIGHT (Chronological Split) MAE: {mean_absolute_error(y[cut:], pred_ok):.2f}  <-- Honest reality")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].scatter(X_tr_b['day'], y_tr_b, s=12, alpha=0.5, label='Train')
axes[0].scatter(X_te_b['day'], y_te_b, s=22, color='orange', label='Test True')
axes[0].scatter(X_te_b['day'], pred_bad, s=20, color='red', marker='x', label='Predicted')
axes[0].set_title('WRONG: Random Split')
axes[0].legend()

axes[1].plot(days[:cut], y[:cut], label='Train (Past)')
axes[1].plot(days[cut:], y[cut:], color='orange', label='Test (Future)')
axes[1].plot(days[cut:], pred_ok, color='red', linestyle='--', lw=2, label='Predicted')
axes[1].set_title('RIGHT: Chronological Split')
axes[1].legend()
plt.tight_layout()
plt.show()

## Rule 3: How to Improve a Model Step-by-Step (Without Guessing)

Whether you have a supervised target `y` (Teams 1, 2, 4, 5) or clustering + scoring (Team 3), follow the same 4-step loop:
1.  **Log a Dumb Benchmark first:** `DummyClassifier` (most frequent class), **Persistence** (`next hour = now`), **Buy-and-Hold**, or raw **`TF-IDF + KMeans`** (Team 3).
2.  **Build 1 Simple Baseline:** One clean `Pipeline` (e.g., `LogisticRegression` or `Ridge`), and record its score.
3.  **Validate Properly (Sprint 4):** One split can be lucky! Use **`TimeSeriesSplit`** (time series), **`StratifiedKFold`** (CEFR text), or **Multi-Topic Golden Set** (Team 3).
4.  **Inspect Errors Before Tuning:** Read your 5 worst mistakes (or 3 worst-ranked pages for Team 3). Ask *"What signal is missing?"* - then add 1 new feature or model at a time, and keep it **only** if your validation score improves!

In [ ]:
# TINY VISUAL DEMO: How TimeSeriesSplit Works (Always Train on Past -> Test on Future)
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
fig, ax = plt.subplots(figsize=(9, 3))
for fold, (tr_i, va_i) in enumerate(tscv.split(np.arange(100))):
    ax.scatter(tr_i, [fold] * len(tr_i), c='#1f77b4', marker='_', lw=10, label='Train (Past)' if fold == 0 else "")
    ax.scatter(va_i, [fold] * len(va_i), c='#ff7f0e', marker='_', lw=10, label='Validation (Future)' if fold == 0 else "")
ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Time (Days / Hours)')
ax.set_title('TimeSeriesSplit: 5 Chronological Checks Instead of 1 Lucky Split')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Roadmap Across All 5 Teams (Work at Your Own Pace)

| Team | Step 1: Quick Data Check | Step 2: First Honest Baseline | Step 3: Sprint 4 Upgrade (When Ready) |
| :--- | :--- | :--- | :--- |
| **Team 1** (World Stocks) | Check `groupby('Ticker').shift(-1)` & Class 3 count | Dummy vs. 1 simple classifier (`Macro F1`) | `TimeSeriesSplit` + your own indicators + `HistGradientBoosting` |
| **Team 2** (Poznan Air Quality) | Check CSV headers/extensions & `ffill(limit=2)` | Persistence vs. Ridge on `pm10` (`MAE/RMSE`) | `TimeSeriesSplit` + cyclical `hour` + lags + PM2.5 |
| **Team 3** (Topic Research & Mind Map) | Human-verify sample + reproducible data | Fill `results.csv` (`TF-IDF` vs. `LSA`) | Your own multi-signal Credibility Scorer + auto node labels |
| **Team 4** (CEFR Difficulty) | Check duplicates & A1 scope decision | Replace stub with `TF-IDF + LogReg` (`Macro F1`) | Plug `cefrwords.json` + linguistic features into `Pipeline` |
| **Team 5** (S&P 500 & Backtesting) | Clean fetch script + frozen CSV | Directional Baseline + Buy-and-Hold curve | Transaction costs + Sharpe Ratio + `TimeSeriesSplit` |

Open your team's guide notebook on Discord. These notebooks teach the **concepts** on tiny toy examples - **you** write the real solution on your project data!